In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import GradientBoostingRegressor
from ipywidgets import interact, Dropdown, IntSlider

In [2]:
# Load processed datasets
train_features = pd.read_csv("../data/processed/train_features.csv")
hourly_df = pd.read_csv("../data/processed/hourly_demand.csv")

# Convert time columns
train_features["hour"] = pd.to_datetime(train_features["hour"], utc=True, errors="coerce")
hourly_df["hour"] = pd.to_datetime(hourly_df["hour"], utc=True, errors="coerce")

print("train_features:", train_features.shape)
print("hourly_df:", hourly_df.shape)
hourly_df.tail()

train_features: (20997, 17)
hourly_df: (23687, 7)


,hour,session_count,total_kwh,hour_of_day,day_of_week,month,is_weekend
23682,2021-09-13 21:00:00+00:00,2,9.000,21,0,9,0
23683,2021-09-13 22:00:00+00:00,1,17.720,22,0,9,0
23684,2021-09-13 23:00:00+00:00,1,2.018,23,0,9,0
23685,2021-09-14 00:00:00+00:00,0,0.000,0,1,9,0
23686,2021-09-14 01:00:00+00:00,1,45.064,1,1,9,0


In [3]:
# Final feature sets

session_feature_cols = [
    "hour_of_day",
    "day_of_week",
    "month",
    "is_weekend",
    "session_lag_1",
    "session_lag_2",
    "session_lag_24",
    "session_lag_168",
    "session_rolling_mean_24",
]

kwh_feature_cols = [
    "hour_of_day",
    "day_of_week",
    "month",
    "is_weekend",
    "session_lag_1",
    "session_lag_2",
    "session_lag_24",
    "session_lag_168",
    "session_rolling_mean_24",
    "kwh_lag_1",
    "kwh_lag_2",
    "kwh_lag_24",
    "kwh_lag_168",
    "kwh_rolling_mean_24",
]

In [4]:
# Train final Gradient Boosting models for forward forecasting

X_train_session = train_features[session_feature_cols]
y_train_session = train_features["session_count"]

X_train_kwh = train_features[kwh_feature_cols]
y_train_kwh = train_features["total_kwh"]

session_gb = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42,
)
session_gb.fit(X_train_session, y_train_session)

kwh_gb = GradientBoostingRegressor(
    n_estimators=200,
    learning_rate=0.05,
    max_depth=3,
    random_state=42,
)
kwh_gb.fit(X_train_kwh, y_train_kwh)

print("Forecasting models trained.")

Forecasting models trained.


In [5]:
def build_session_feature_row(timestamp: pd.Timestamp, history_df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "hour_of_day": [timestamp.hour],
        "day_of_week": [timestamp.dayofweek],
        "month": [timestamp.month],
        "is_weekend": [1 if timestamp.dayofweek in [5, 6] else 0],
        "session_lag_1": [history_df["session_count"].iloc[-1]],
        "session_lag_2": [history_df["session_count"].iloc[-2]],
        "session_lag_24": [history_df["session_count"].iloc[-24]],
        "session_lag_168": [history_df["session_count"].iloc[-168]],
        "session_rolling_mean_24": [history_df["session_count"].iloc[-24:].mean()],
    })


def build_kwh_feature_row(timestamp: pd.Timestamp, history_df: pd.DataFrame) -> pd.DataFrame:
    return pd.DataFrame({
        "hour_of_day": [timestamp.hour],
        "day_of_week": [timestamp.dayofweek],
        "month": [timestamp.month],
        "is_weekend": [1 if timestamp.dayofweek in [5, 6] else 0],
        "session_lag_1": [history_df["session_count"].iloc[-1]],
        "session_lag_2": [history_df["session_count"].iloc[-2]],
        "session_lag_24": [history_df["session_count"].iloc[-24]],
        "session_lag_168": [history_df["session_count"].iloc[-168]],
        "session_rolling_mean_24": [history_df["session_count"].iloc[-24:].mean()],
        "kwh_lag_1": [history_df["total_kwh"].iloc[-1]],
        "kwh_lag_2": [history_df["total_kwh"].iloc[-2]],
        "kwh_lag_24": [history_df["total_kwh"].iloc[-24]],
        "kwh_lag_168": [history_df["total_kwh"].iloc[-168]],
        "kwh_rolling_mean_24": [history_df["total_kwh"].iloc[-24:].mean()],
    })

In [6]:
def recursive_forecast(history_df: pd.DataFrame, forecast_start: pd.Timestamp, horizon_hours: int) -> pd.DataFrame:
    history = history_df.copy().reset_index(drop=True)
    forecast_rows = []

    for step in range(horizon_hours):
        current_time = forecast_start + pd.Timedelta(hours=step)

        X_session = build_session_feature_row(current_time, history)
        pred_session = session_gb.predict(X_session)[0]

        X_kwh = build_kwh_feature_row(current_time, history)
        pred_kwh = kwh_gb.predict(X_kwh)[0]

        forecast_rows.append({
            "hour": current_time,
            "predicted_session_count": pred_session,
            "predicted_total_kwh": pred_kwh,
            "day_name": current_time.day_name(),
            "hour_of_day": current_time.hour,
        })

        next_row = pd.DataFrame({
            "hour": [current_time],
            "session_count": [pred_session],
            "total_kwh": [pred_kwh],
        })

        history = pd.concat([history, next_row], ignore_index=True)

    return pd.DataFrame(forecast_rows)

In [10]:
# Presentation-oriented future start options
future_start_options = [
    "2026-03-30 00:00:00+00:00",
    "2026-03-30 06:00:00+00:00",
    "2026-03-30 12:00:00+00:00",
    "2026-03-31 00:00:00+00:00",
    "2026-03-31 12:00:00+00:00",
    "2026-04-01 00:00:00+00:00",
]

In [13]:
def forecast_view(future_start=None, horizon_hours=24, history_window=168):
    future_start = pd.to_datetime(future_start, utc=True)

    # Use the most recent observed history from the dataset as the known history
    seed_history = hourly_df.iloc[-history_window:].copy()

    forecast_df = recursive_forecast(
        history_df=seed_history[["hour", "session_count", "total_kwh"]],
        forecast_start=future_start,
        horizon_hours=horizon_hours,
    )

    # Prevent impossible negative forecasts
    forecast_df["predicted_session_count"] = forecast_df["predicted_session_count"].clip(lower=0)
    forecast_df["predicted_total_kwh"] = forecast_df["predicted_total_kwh"].clip(lower=0)

    # Build a cleaner presentation table
    forecast_table = forecast_df.copy()
    forecast_table["Forecast Session Count"] = forecast_table["predicted_session_count"].round(2)
    forecast_table["Forecast Total kWh"] = forecast_table["predicted_total_kwh"].round(2)

    forecast_end = future_start + pd.Timedelta(hours=horizon_hours - 1)

    print("Forecast start:", future_start)
    print("Forecast end:", forecast_end)
    print("Forecast horizon:", horizon_hours, "hours")
    print()

    display(
        forecast_table[
            ["hour", "Forecast Session Count", "Forecast Total kWh", "day_name", "hour_of_day"]
        ]
    )

    history_x = np.arange(-history_window, 0)
    forecast_x = np.arange(0, horizon_hours)

    plt.figure(figsize=(12, 5))
    plt.plot(
        history_x,
        seed_history["session_count"].values,
        label="Observed History",
        linewidth=2
    )
    plt.plot(
        forecast_x,
        forecast_df["predicted_session_count"].values,
        label="Forecast",
        linewidth=2
    )
    plt.axvline(0, linestyle="--")
    plt.title(f"Forward Forecast for Session Count Starting {future_start}")
    plt.xlabel("Hours from Forecast Start")
    plt.ylabel("Session Count")
    plt.legend()
    plt.tight_layout()
    plt.show()

    plt.figure(figsize=(12, 5))
    plt.plot(
        history_x,
        seed_history["total_kwh"].values,
        label="Observed History",
        linewidth=2
    )
    plt.plot(
        forecast_x,
        forecast_df["predicted_total_kwh"].values,
        label="Forecast",
        linewidth=2
    )
    plt.axvline(0, linestyle="--")
    plt.title(f"Forward Forecast for Total kWh Starting {future_start}")
    plt.xlabel("Hours from Forecast Start")
    plt.ylabel("Total kWh")
    plt.legend()
    plt.tight_layout()
    plt.show()

In [14]:
interact(
    forecast_view,
    future_start=Dropdown(
        options=future_start_options,
        value="2026-03-30 00:00:00+00:00",
        description="Start"
    ),
    horizon_hours=IntSlider(
        value=24,
        min=6,
        max=48,
        step=6,
        description="Hours"
    ),
    history_window=IntSlider(
        value=168,
        min=48,
        max=336,
        step=24,
        description="History"
    ),
);

interactive(children=(Dropdown(description='Start', options=('2026-03-30 00:00:00+00:00', '2026-03-30 06:00:00…